# 06 · Propuesta Conceptual — Capa GOLD: Modelo Estrella (TUNI.pe / UCSP)

**Objetivo:** diseñar la propuesta conceptual del **modelo estrella** para la capa **Gold**, a partir de los datos limpios de la capa Silver. Este notebook es de **diseño**: no genera archivos Gold.

**Entradas:**
- `data/Silver/ingresantes_clean_v2.parquet` (~3.12 M filas, 27 cols) — Ingresantes **anuales** (2020–2025).
- `data/Silver/matriculados_clean_v2.parquet` (~17.37 M filas, 34 cols) — Matriculados **semestrales** (2020-1 … 2025-2).

**Salidas:** propuesta conceptual (dimensiones, hechos, diagrama, agregaciones de ejemplo y validación). Sin escritura de archivos Gold.

**Decisiones de diseño adoptadas:**
- `DimPeriodo` única y conforme: `SEMESTRE` **nullable** (Ingresantes es anual → `SEMESTRE = NULL`; Matriculados semestral → 1/2).
- `DimUbicacion` conforme a nivel **departamento + provincia** (ambos datasets la tienen; Ingresantes no tiene distrito).
- `DimUniversidad`: unificación de `LICENCIADO` (Ingresantes) y `LICENCIA` (Matriculados) → `ESTADO_LICENCIAMIENTO`.
- Entrada de Matriculados: `matriculados_clean_v2.parquet`.

**Técnica:** catálogos de dimensiones construidos con `scan_parquet` + `unique()` (streaming, sin OOM); análisis sobre muestras (Matriculados: 100,000 filas estratificadas por periodo).

In [1]:
# Configuración: límite de hilos ANTES de importar Polars (12 núcleos)
import os
os.environ['POLARS_MAX_THREADS'] = '12'

import gc
from pathlib import Path

import polars as pl

pl.Config.set_streaming_chunk_size(32 * 1024 * 1024)  # 32 MB por lote de streaming

print('polars', pl.__version__)
print('hilos activos:', pl.thread_pool_size())


def rss_actual_gb():
    '''RSS actual del proceso en GB (Linux, /proc/self/statm).'''
    try:
        with open('/proc/self/statm', encoding='utf-8') as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf('SC_PAGE_SIZE') / (1024**3)
    except (OSError, ValueError, IndexError):
        return float('nan')


print(f'RSS inicial: {rss_actual_gb():.2f} GB')
print()

# Rutas del proyecto (misma detección automática que los notebooks 01-05)
current_dir = Path.cwd()
if (current_dir / 'data').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'data').exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError('No se encontró la carpeta data. Ejecuta desde la raíz o desde notebooks/.')

SILVER = PROJECT_ROOT / 'data' / 'Silver'
ING_V2 = SILVER / 'ingresantes_clean_v2.parquet'
MAT_V2 = SILVER / 'matriculados_clean_v2.parquet'

assert ING_V2.exists(), f'No existe {ING_V2.name}. Ejecuta primero 04_limpieza_ingresantes.ipynb.'
assert MAT_V2.exists(), f'No existe {MAT_V2.name}. Ejecuta primero 04_limpieza_matriculados.ipynb.'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Ingresantes V2:', ING_V2)
print('Matriculados V2:', MAT_V2)

polars 1.44.1
hilos activos: 12
RSS inicial: 0.08 GB

PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
Ingresantes V2: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/ingresantes_clean_v2.parquet
Matriculados V2: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/matriculados_clean_v2.parquet


---
## SECCIÓN 1 · Carga de muestras (sin OOM)

- **Ingresantes** (3.12 M filas, ~0.14 GB): cabe en RAM → se carga completo con `read_parquet`.
- **Matriculados** (17.37 M filas, ~736 MB): se carga una muestra **estratificada por periodo** (n≈100,000) con `scan_parquet` + `sample()` por semestre, garantizando que el modelo refleje **todas las temporalidades** (2020-1 … 2025-2).

Se muestran `shape` y tipos de datos de ambas muestras.

In [3]:
# Ingresantes: dataset pequeño → carga completa (cabe en RAM)
ing_muestra = pl.read_parquet(ING_V2)

# Matriculados: muestra estratificada por periodo (n total ≈ 100,000), determinista
N_MUESTRA = 100_000
SEED = 42

dist_periodo = (
    pl.scan_parquet(MAT_V2)
    .select('PERIODO_ESTANDARIZADO')
    .group_by('PERIODO_ESTANDARIZADO')
    .len()
    .collect()
    .with_columns(
        (pl.col('len') / pl.col('len').sum() * N_MUESTRA).round().cast(pl.Int64).alias('n')
    )
)

# ⚠️ CORRECCIÓN: .collect() ANTES de .sample() + .sample(fraction=1.0, shuffle=True) en lugar de .shuffle()
mat_muestra = pl.concat([
    pl.scan_parquet(MAT_V2)
    .filter(pl.col('PERIODO_ESTANDARIZADO') == p)
    .collect()
    .sample(n=n_p, seed=SEED, with_replacement=False)
    for p, n_p in zip(dist_periodo['PERIODO_ESTANDARIZADO'], dist_periodo['n'])
]).sample(fraction=1.0, shuffle=True, seed=SEED)

print('Muestra Ingresantes:', ing_muestra.shape)
print('Muestra Matriculados:', mat_muestra.shape, '(objetivo 100,000)')
print()
print('Distribución por periodo (muestra estratificada):')
print(mat_muestra.group_by('PERIODO_ESTANDARIZADO').len().sort('PERIODO_ESTANDARIZADO'))
print()
print('Esquema Ingresantes:')
print(ing_muestra.schema)
print()
print('Esquema Matriculados:')
print(mat_muestra.schema)

Muestra Ingresantes: (3119994, 27)
Muestra Matriculados: (99999, 34) (objetivo 100,000)

Distribución por periodo (muestra estratificada):
shape: (12, 2)
┌───────────────────────┬──────┐
│ PERIODO_ESTANDARIZADO ┆ len  │
│ ---                   ┆ ---  │
│ str                   ┆ u32  │
╞═══════════════════════╪══════╡
│ 2020-1                ┆ 7056 │
│ 2020-2                ┆ 6827 │
│ 2021-1                ┆ 7940 │
│ 2021-2                ┆ 7935 │
│ 2022-1                ┆ 8530 │
│ …                     ┆ …    │
│ 2023-2                ┆ 8147 │
│ 2024-1                ┆ 9021 │
│ 2024-2                ┆ 8840 │
│ 2025-1                ┆ 9654 │
│ 2025-2                ┆ 9321 │
└───────────────────────┴──────┘

Esquema Ingresantes:
Schema({'CODIGO_INEI': String, 'NOMBRE_ENTIDAD': String, 'TIPO_ENTIDAD': String, 'TIPO_GESTION': String, 'LICENCIADO': String, 'TIPO_CONSTITUCION': String, 'NIVEL_ACADEMICO': String, 'PROCESO_ESTANDARIZADO': Int64, 'GUID_PERSONA': String, 'SEXO': String, 'NACIONA

---
## SECCIÓN 2 · Estructura y claves

Claves naturales documentadas (definidas y validadas en los notebooks 04 y 05):

- **Ingresantes (CLAVE_ING):** `CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA` → 0 duplicados. Una fila = un ingresante × programa × universidad.
- **Matriculados (K5):** `PERIODO_ESTANDARIZADO + CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_LOCAL + GUID_PERSONA` → clave natural; en el V2 persisten duplicados exactos que se resolverán en Gold.

Columnas candidatas a dimensiones (agrupadas por dominio):

| Dimensión | Columnas (Ingresantes / Matriculados) |
|---|---|
| Universidad | CODIGO_INEI, NOMBRE_ENTIDAD, TIPO_ENTIDAD, TIPO_GESTION, LICENCIADO/LICENCIA, TIPO_CONSTITUCION |
| Programa | CODIGO_SIU_PROGRAMA, NOMBRE_PROGRAMA, CODIGO_GRUPO_1/3, NOMBRE_GRUPO_1/3, NIVEL_ACADEMICO |
| Periodo | PROCESO_ESTANDARIZADO (anual) / PERIODO_ESTANDARIZADO (semestral) |
| Ubicación | DEPARTAMENTO_FILIAL + PROVINCIA_FILIAL / DEPARTAMENTO_LOCAL + PROVINCIA_LOCAL + DISTRITO_LOCAL |
| Local | — / CODIGO_LOCAL, ES_LOCAL_PRINCIPAL, CODIGO_UBIGEO_INEI_LOCAL |
| Persona (degenerada) | GUID_PERSONA, SEXO, EDAD, NACIONALIDAD, DEPARTAMENTO_NACIMIENTO, ANIO_NACIMIENTO, CERT_GRAVEDAD, TIENE_DISCAPACIDAD |

In [4]:
# Claves naturales documentadas (notebooks 04 y 05)
CLAVE_ING = ['CODIGO_INEI', 'GUID_PERSONA', 'CODIGO_SIU_PROGRAMA']
CLAVE_MAT = ['PERIODO_ESTANDARIZADO', 'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_LOCAL', 'GUID_PERSONA']


def contar_dups(df, key):
    return df.group_by(key).len().filter(pl.col('len') > 1).select(pl.col('len').sum()).item()


print('Unicidad de claves naturales SOBRE LAS MUESTRAS (indicativa):')
print(f'  Ingresantes  CLAVE_ING: {contar_dups(ing_muestra, CLAVE_ING):,} duplicados')
print(f'  Matriculados CLAVE_MAT: {contar_dups(mat_muestra, CLAVE_MAT):,} duplicados')
print()

print('Columnas candidatas a dimensiones (por dominio):')
print('  Universidad (ING/MAT): CODIGO_INEI, NOMBRE_ENTIDAD, TIPO_ENTIDAD, TIPO_GESTION,')
print('                        LICENCIADO(ING)/LICENCIA(MAT), TIPO_CONSTITUCION')
print('  Programa (ING/MAT):    CODIGO_SIU_PROGRAMA, NOMBRE_PROGRAMA, CODIGO_GRUPO_1/3, NOMBRE_GRUPO_1/3, NIVEL_ACADEMICO')
print('  Periodo:               PROCESO_ESTANDARIZADO (ING, anual) / PERIODO_ESTANDARIZADO (MAT, semestral)')
print('  Ubicacion (ING/MAT):   DEPARTAMENTO_FILIAL+PROVINCIA_FILIAL / DEPARTAMENTO_LOCAL+PROVINCIA_LOCAL+DISTRITO_LOCAL')
print('  Local (solo MAT):      CODIGO_LOCAL, ES_LOCAL_PRINCIPAL, CODIGO_UBIGEO_INEI_LOCAL')
print('  Persona (degenerada):  GUID_PERSONA, SEXO, EDAD, NACIONALIDAD, DEPARTAMENTO_NACIMIENTO,')
print('                        ANIO_NACIMIENTO, CERT_GRAVEDAD, TIENE_DISCAPACIDAD')

Unicidad de claves naturales SOBRE LAS MUESTRAS (indicativa):
  Ingresantes  CLAVE_ING: 0 duplicados
  Matriculados CLAVE_MAT: 0 duplicados

Columnas candidatas a dimensiones (por dominio):
  Universidad (ING/MAT): CODIGO_INEI, NOMBRE_ENTIDAD, TIPO_ENTIDAD, TIPO_GESTION,
                        LICENCIADO(ING)/LICENCIA(MAT), TIPO_CONSTITUCION
  Programa (ING/MAT):    CODIGO_SIU_PROGRAMA, NOMBRE_PROGRAMA, CODIGO_GRUPO_1/3, NOMBRE_GRUPO_1/3, NIVEL_ACADEMICO
  Periodo:               PROCESO_ESTANDARIZADO (ING, anual) / PERIODO_ESTANDARIZADO (MAT, semestral)
  Ubicacion (ING/MAT):   DEPARTAMENTO_FILIAL+PROVINCIA_FILIAL / DEPARTAMENTO_LOCAL+PROVINCIA_LOCAL+DISTRITO_LOCAL
  Local (solo MAT):      CODIGO_LOCAL, ES_LOCAL_PRINCIPAL, CODIGO_UBIGEO_INEI_LOCAL
  Persona (degenerada):  GUID_PERSONA, SEXO, EDAD, NACIONALIDAD, DEPARTAMENTO_NACIMIENTO,
                        ANIO_NACIMIENTO, CERT_GRAVEDAD, TIENE_DISCAPACIDAD


---
## SECCIÓN 3 · Propuesta de modelo estrella (Gold)

### Dimensiones

| Dimensión | Clave natural | Atributos principales |
|---|---|---|
| DimUniversidad | CODIGO_INEI | NOMBRE_ENTIDAD, TIPO_ENTIDAD, TIPO_GESTION, TIPO_CONSTITUCION, ESTADO_LICENCIAMIENTO |
| DimPrograma | CODIGO_SIU_PROGRAMA | NOMBRE_PROGRAMA, CODIGO_GRUPO_1, NOMBRE_GRUPO_1, CODIGO_GRUPO_3, NOMBRE_GRUPO_3, NIVEL_ACADEMICO |
| DimPeriodo | ANIO + SEMESTRE (NULL = anual) | LABEL_PERIODO (ej. 2025-1 / 2025), TIPO_PERIODO (SEMESTRAL / ANUAL) |
| DimUbicacion | DEPARTAMENTO + PROVINCIA | Region_Sur (booleano, derivado del departamento) |
| DimLocal (opcional, MAT) | CODIGO_LOCAL | DEPARTAMENTO_LOCAL, PROVINCIA_LOCAL, DISTRITO_LOCAL, ES_LOCAL_PRINCIPAL, CODIGO_UBIGEO_INEI_LOCAL |

### Tablas de hechos

| Hecho | Granularidad | Medidas / atributos degenerados | FKs |
|---|---|---|---|
| FactIngresantes | 1 fila = ingresante × programa × universidad × año | GUID_PERSONA (conteo), SEXO, EDAD, NACIONALIDAD, Region_Sur | FK_Universidad, FK_Programa, FK_Periodo, FK_Ubicacion |
| FactMatriculados | 1 fila = matrícula (persona × programa × local × semestre) | GUID_PERSONA (conteo), SEXO, EDAD, NACIONALIDAD, Region_Sur | FK_Universidad, FK_Programa, FK_Periodo, FK_Ubicacion, FK_Local |

**Notas de diseño:**
- `GUID_PERSONA` se usa como medida de conteo (`COUNT(DISTINCT GUID_PERSONA)`); `SEXO`, `EDAD`, `NACIONALIDAD` y `Region_Sur` quedan como **atributos degenerados** en el hecho (denormalización deliberada para evitar joins en filtros frecuentes del caso de negocio).
- `Region_Sur` también vive en `DimUbicacion`; mantenerla en el hecho es una decisión de **rendimiento** (redundancia controlada).
- Las claves naturales se convierten directamente en **FK** hacia las dimensiones; en implementación Gold se añadirán claves surrogate (`SK_*`).

In [5]:
# --- Catálogos de dimensiones EXACTOS vía scan_parquet + unique (sin OOM) ---
DEPARTAMENTOS_SUR = ['AREQUIPA', 'CUSCO', 'TACNA', 'PUNO', 'MOQUEGUA', 'APURIMAC']

# DimUniversidad: unión ING (LICENCIADO) + MAT (LICENCIA) → normalización a ESTADO_LICENCIAMIENTO
dim_univ = (
    pl.concat([
        pl.scan_parquet(ING_V2)
        .select(['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'LICENCIADO', 'TIPO_CONSTITUCION'])
        .rename({'LICENCIADO': 'LICENCIA'})
        .unique()
        .collect(),
        pl.scan_parquet(MAT_V2)
        .select(['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'LICENCIA', 'TIPO_CONSTITUCION'])
        .unique()
        .collect(),
    ])
    .unique()
    .sort('CODIGO_INEI')
    .with_columns(
        pl.when(pl.col('LICENCIA') == 'LICENCIADA').then(pl.lit('LICENCIADO'))
        .when(pl.col('LICENCIA') == 'LEY DE CREACION').then(pl.lit('LEY DE CREACION'))
        .otherwise(pl.lit('NO LICENCIADO'))
        .alias('ESTADO_LICENCIAMIENTO')
    )
    .drop('LICENCIA')
)

# DimPrograma: unión ING + MAT
cols_prog = ['CODIGO_SIU_PROGRAMA', 'NOMBRE_PROGRAMA', 'CODIGO_GRUPO_1', 'NOMBRE_GRUPO_1', 'CODIGO_GRUPO_3', 'NOMBRE_GRUPO_3', 'NIVEL_ACADEMICO']
dim_prog = pl.concat([
    pl.scan_parquet(ING_V2).select(cols_prog).unique().collect(),
    pl.scan_parquet(MAT_V2).select(cols_prog).unique().collect(),
]).unique().sort('CODIGO_SIU_PROGRAMA')

# DimPeriodo: semestres (MAT) + anuales (ING, SEMESTRE = NULL)
dim_periodo_sem = (
    pl.scan_parquet(MAT_V2).select('PERIODO_ESTANDARIZADO').unique().collect()
    .with_columns([
        pl.col('PERIODO_ESTANDARIZADO').str.slice(0, 4).cast(pl.Int64).alias('ANIO'),
        pl.col('PERIODO_ESTANDARIZADO').str.slice(5, 1).cast(pl.Int64).alias('SEMESTRE'),
        pl.lit('SEMESTRAL').alias('TIPO_PERIODO'),
    ])
    .select(['ANIO', 'SEMESTRE', pl.col('PERIODO_ESTANDARIZADO').alias('LABEL_PERIODO'), 'TIPO_PERIODO'])
)
dim_periodo_an = (
    pl.scan_parquet(ING_V2).select('PROCESO_ESTANDARIZADO').unique().collect()
    .with_columns([
        pl.col('PROCESO_ESTANDARIZADO').cast(pl.Int64).alias('ANIO'),
        pl.lit(None, dtype=pl.Int64).alias('SEMESTRE'),
        pl.col('PROCESO_ESTANDARIZADO').cast(pl.String).alias('LABEL_PERIODO'),
        pl.lit('ANUAL').alias('TIPO_PERIODO'),
    ])
    .select(['ANIO', 'SEMESTRE', 'LABEL_PERIODO', 'TIPO_PERIODO'])
)
dim_periodo = pl.concat([dim_periodo_sem, dim_periodo_an]).sort('ANIO', 'SEMESTRE').with_row_index('SK_Periodo', offset=1)

# DimUbicacion: conforme DEPARTAMENTO + PROVINCIA (FILIAL en ING, LOCAL en MAT)
dim_ubicacion = (
    pl.concat([
        pl.scan_parquet(ING_V2).select([
            pl.col('DEPARTAMENTO_FILIAL').alias('DEPARTAMENTO'),
            pl.col('PROVINCIA_FILIAL').alias('PROVINCIA'),
        ]).unique().collect(),
        pl.scan_parquet(MAT_V2).select([
            pl.col('DEPARTAMENTO_LOCAL').alias('DEPARTAMENTO'),
            pl.col('PROVINCIA_LOCAL').alias('PROVINCIA'),
        ]).unique().collect(),
    ])
    .unique()
    .sort('DEPARTAMENTO', 'PROVINCIA')
    .with_columns(pl.col('DEPARTAMENTO').is_in(DEPARTAMENTOS_SUR).alias('Region_Sur'))
)

# DimLocal: solo Matriculados
dim_local = (
    pl.scan_parquet(MAT_V2)
    .select(['CODIGO_LOCAL', 'DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL', 'DISTRITO_LOCAL', 'ES_LOCAL_PRINCIPAL', 'CODIGO_UBIGEO_INEI_LOCAL'])
    .unique()
    .collect()
    .sort('CODIGO_LOCAL')
)

print('CATÁLOGOS DE DIMENSIÓN (exactos, vía scan_parquet + unique — sin OOM)')
print()
for nombre, df, clave in [
    ('DimUniversidad', dim_univ, 'CODIGO_INEI'),
    ('DimPrograma', dim_prog, 'CODIGO_SIU_PROGRAMA'),
    ('DimPeriodo', dim_periodo, 'ANIO + SEMESTRE (NULL=anual)'),
    ('DimUbicacion', dim_ubicacion, 'DEPARTAMENTO + PROVINCIA'),
    ('DimLocal (MAT)', dim_local, 'CODIGO_LOCAL'),
]:
    print(f'--- {nombre} · {df.height:,} filas · clave natural: {clave}')
    print(df.head(5))
    print()

print('Mapeo CLAVE NATURAL -> FK:')
print('  FactIngresantes:  FK_Universidad=CODIGO_INEI | FK_Programa=CODIGO_SIU_PROGRAMA | FK_Periodo=ANIO | FK_Ubicacion=DEPARTAMENTO+PROVINCIA')
print('  FactMatriculados: FK_Universidad=CODIGO_INEI | FK_Programa=CODIGO_SIU_PROGRAMA | FK_Periodo=ANIO+SEMESTRE | FK_Ubicacion=DEPARTAMENTO+PROVINCIA | FK_Local=CODIGO_LOCAL')

CATÁLOGOS DE DIMENSIÓN (exactos, vía scan_parquet + unique — sin OOM)

--- DimUniversidad · 177 filas · clave natural: CODIGO_INEI
shape: (5, 6)
┌─────────────┬──────────────────┬──────────────┬──────────────┬─────────────────┬─────────────────┐
│ CODIGO_INEI ┆ NOMBRE_ENTIDAD   ┆ TIPO_ENTIDAD ┆ TIPO_GESTION ┆ TIPO_CONSTITUCI ┆ ESTADO_LICENCIA │
│ ---         ┆ ---              ┆ ---          ┆ ---          ┆ ON              ┆ MIENTO          │
│ str         ┆ str              ┆ str          ┆ str          ┆ ---             ┆ ---             │
│             ┆                  ┆              ┆              ┆ str             ┆ str             │
╞═════════════╪══════════════════╪══════════════╪══════════════╪═════════════════╪═════════════════╡
│ 160000001   ┆ UNIVERSIDAD      ┆ UNIVERSIDAD  ┆ PUBLICO      ┆ PUBLICA         ┆ LICENCIADO      │
│             ┆ NACIONAL MAYOR   ┆              ┆              ┆                 ┆                 │
│             ┆ DE …             ┆             

---
## SECCIÓN 4 · Justificación y diagrama del esquema estrella

### ¿Por qué cada dimensión?

- **DimUniversidad**: los hechos miden ingresantes/matrículas **por institución**; el benchmark UCSP vs nacional y el análisis por gestión/tipo/licenciamiento se apoyan en sus atributos.
- **DimPrograma**: el análisis por **carrera** (top 5 en región sur) necesita la jerarquía de grupos de carrera (`CODIGO_GRUPO_1` / `CODIGO_GRUPO_3`).
- **DimPeriodo**: separa la temporalidad en una dimensión conforme: anual para Ingresantes, semestral para Matriculados, con `SEMESTRE` nullable y clave `ANIO + SEMESTRE`.
- **DimUbicacion**: agrupa por **departamento/provincia** de la sede (FILIAL en Ingresantes, LOCAL en Matriculados) y es la fuente de `Region_Sur`.
- **DimLocal**: detalle del **local** (distrito, ubicación) que solo existe en Matriculados; sin ella se perdería la granularidad distrital.

### Claves naturales → claves foráneas

| Hecho | FK | Clave natural fuente |
|---|---|---|
| Ambos | FK_Universidad | CODIGO_INEI |
| Ambos | FK_Programa | CODIGO_SIU_PROGRAMA |
| Ambos | FK_Periodo | ANIO (+ SEMESTRE si aplica) |
| Ambos | FK_Ubicacion | DEPARTAMENTO + PROVINCIA |
| FactMatriculados | FK_Local | CODIGO_LOCAL |

### Diagrama ASCII

```
                    MODELO ESTRELLA — CAPA GOLD

   DIMENSIONES                         HECHOS (granularidad 1:N)
   -----------                         --------------------------
   DimUniversidad            <-----    FactIngresantes
     PK CODIGO_INEI                     FK_Universidad -> DimUniversidad.CODIGO_INEI
     NOMBRE_ENTIDAD                     FK_Programa    -> DimPrograma.CODIGO_SIU_PROGRAMA
     TIPO_ENTIDAD                       FK_Periodo     -> DimPeriodo (anual)
     TIPO_GESTION                       FK_Ubicacion   -> DimUbicacion (DEPT+PROV)
     ESTADO_LICENCIAMIENTO              GUID_PERSONA (conteo), SEXO, EDAD,
                                        NACIONALIDAD, Region_Sur
   DimPrograma               <-----    FactMatriculados
     PK CODIGO_SIU_PROGRAMA             FK_Universidad -> DimUniversidad.CODIGO_INEI
     NOMBRE_PROGRAMA                    FK_Programa    -> DimPrograma.CODIGO_SIU_PROGRAMA
     GRUPO_1 (CODIGO/NOMBRE)            FK_Periodo     -> DimPeriodo (ANIO+SEMESTRE)
     GRUPO_3 (CODIGO/NOMBRE)            FK_Ubicacion   -> DimUbicacion (DEPT+PROV)
                                        FK_Local       -> DimLocal.CODIGO_LOCAL
   DimPeriodo                <-----     GUID_PERSONA (conteo), SEXO, EDAD,
     PK ANIO + SEMESTRE (NULL=anual)    NACIONALIDAD, Region_Sur
     LABEL_PERIODO, TIPO_PERIODO
                                        DimLocal (solo FactMatriculados)
   DimUbicacion               <-----      PK CODIGO_LOCAL
     PK DEPARTAMENTO + PROVINCIA         DEPARTAMENTO_LOCAL / PROVINCIA_LOCAL /
     Region_Sur                          DISTRITO_LOCAL, ES_LOCAL_PRINCIPAL,
                                        CODIGO_UBIGEO_INEI_LOCAL

   Notas:
     - Todas las aristas son DIMENSION -> HECHO (1:N).
     - Sin aristas entre dimensiones ni entre hechos -> sin ciclos.
     - DimLocal solo se conecta a FactMatriculados.
```

---
## SECCIÓN 5 · Ejemplos de agregaciones

Consultas que el modelo estrella debe responder (se demuestran sobre las **muestras**; en Gold se ejecutarían sobre las tablas completas):

1. **Total de matriculados por universidad y año** — benchmark de tamaño por institución y temporalidad.
2. **Top 5 carreras en la región sur** — el caso de negocio (nivel sur del país).
3. **Comparativa UCSP vs nacional** — participación de la UCSP en el total nacional.

Equivalente en SQL sobre el modelo estrella (ejemplo):

```sql
SELECT u.NOMBRE_ENTIDAD, p.ANIO, COUNT(*) AS total_matriculas
FROM FactMatriculados f
JOIN DimUniversidad u ON f.FK_Universidad = u.CODIGO_INEI
JOIN DimPeriodo      p ON f.FK_Periodo    = p.ANIO AND f.FK_SEMESTRE = p.SEMESTRE
GROUP BY u.NOMBRE_ENTIDAD, p.ANIO;
```

In [6]:
print('Ejemplo 1 · Matriculados por universidad y año (muestra, escala ≈1:174):')
mat_por_univ_anio = (
    mat_muestra
    .with_columns(pl.col('PERIODO_ESTANDARIZADO').str.slice(0, 4).alias('ANIO'))
    .group_by('NOMBRE_ENTIDAD', 'ANIO')
    .len()
    .sort('NOMBRE_ENTIDAD', 'ANIO')
)
print(mat_por_univ_anio.head(8))
print()

print('Ejemplo 2 · Top 5 carreras en la región sur (muestra):')
top_sur = (
    mat_muestra
    .filter(pl.col('Region_Sur'))
    .group_by('NOMBRE_PROGRAMA')
    .len()
    .sort('len', descending=True)
    .head(5)
)
print(top_sur)
print()

print('Ejemplo 3 · Comparativa UCSP vs nacional (muestra):')
ucsp = mat_muestra.filter(pl.col('NOMBRE_ENTIDAD') == 'UNIVERSIDAD CATOLICA SAN PABLO').height
nacional = mat_muestra.height
print(f'  Total nacional (muestra):        {nacional:,}')
print(f'  UCSP (UNIVERSIDAD CATOLICA SAN PABLO): {ucsp:,}  ({ucsp / nacional:.2%} del total)')

Ejemplo 1 · Matriculados por universidad y año (muestra, escala ≈1:174):
shape: (8, 3)
┌─────────────────────────────────┬──────┬─────┐
│ NOMBRE_ENTIDAD                  ┆ ANIO ┆ len │
│ ---                             ┆ ---  ┆ --- │
│ str                             ┆ str  ┆ u32 │
╞═════════════════════════════════╪══════╪═════╡
│ ACADEMIA DIPLOMATICA DEL PERU … ┆ 2021 ┆ 1   │
│ ACADEMIA DIPLOMATICA DEL PERU … ┆ 2025 ┆ 1   │
│ ASOCIACION CIVIL UNIVERSIDAD D… ┆ 2020 ┆ 38  │
│ ASOCIACION CIVIL UNIVERSIDAD D… ┆ 2021 ┆ 21  │
│ ASOCIACION CIVIL UNIVERSIDAD D… ┆ 2022 ┆ 35  │
│ ASOCIACION CIVIL UNIVERSIDAD D… ┆ 2023 ┆ 46  │
│ ASOCIACION CIVIL UNIVERSIDAD D… ┆ 2024 ┆ 72  │
│ ASOCIACION CIVIL UNIVERSIDAD D… ┆ 2025 ┆ 74  │
└─────────────────────────────────┴──────┴─────┘

Ejemplo 2 · Top 5 carreras en la región sur (muestra):
shape: (5, 2)
┌───────────────────────┬──────┐
│ NOMBRE_PROGRAMA       ┆ len  │
│ ---                   ┆ ---  │
│ str                   ┆ u32  │
╞═══════════════════════╪

---
## SECCIÓN 6 · Validación conceptual

Se verifican los supuestos del modelo sobre las muestras:

- **A · Integridad referencial:** cada valor de FK presente en los hechos existe en su dimensión (0 huérfanos).
- **B · Cardinalidad 1:N:** cada clave de dimensión se repite en muchas filas de hecho (una dimensión → muchos hechos).
- **C · Sin ciclos ni relaciones innecesarias:** todas las aristas son DIMENSIÓN → HECHO.

In [7]:
print('VALIDACIÓN CONCEPTUAL DEL MODELO ESTRELLA')
print('=' * 78)

# A) Integridad referencial: cada FK del hecho existe en su dimensión (sobre muestras)
print('A) Integridad referencial de las FKs (sobre las muestras):')
mat_ubic = mat_muestra.select([
    pl.col('DEPARTAMENTO_LOCAL').alias('DEPARTAMENTO'),
    pl.col('PROVINCIA_LOCAL').alias('PROVINCIA'),
])
mat_periodo = mat_muestra.with_columns([
    pl.col('PERIODO_ESTANDARIZADO').str.slice(0, 4).cast(pl.Int64).alias('ANIO'),
    pl.col('PERIODO_ESTANDARIZADO').str.slice(5, 1).cast(pl.Int64).alias('SEMESTRE'),
]).select(['ANIO', 'SEMESTRE'])

checks = [
    ('FK_Universidad (CODIGO_INEI)', mat_muestra, ['CODIGO_INEI'], dim_univ, ['CODIGO_INEI']),
    ('FK_Programa (CODIGO_SIU_PROGRAMA)', mat_muestra, ['CODIGO_SIU_PROGRAMA'], dim_prog, ['CODIGO_SIU_PROGRAMA']),
    ('FK_Periodo (ANIO+SEMESTRE)', mat_periodo, ['ANIO', 'SEMESTRE'], dim_periodo, ['ANIO', 'SEMESTRE']),
    ('FK_Ubicacion (DEPT+PROV)', mat_ubic, ['DEPARTAMENTO', 'PROVINCIA'], dim_ubicacion, ['DEPARTAMENTO', 'PROVINCIA']),
    ('FK_Local (CODIGO_LOCAL)', mat_muestra, ['CODIGO_LOCAL'], dim_local, ['CODIGO_LOCAL']),
    ('ING FK_Universidad (CODIGO_INEI)', ing_muestra, ['CODIGO_INEI'], dim_univ, ['CODIGO_INEI']),
    ('ING FK_Periodo (PROCESO_ESTANDARIZADO)', ing_muestra.select(pl.col('PROCESO_ESTANDARIZADO').cast(pl.Int64).alias('ANIO')), ['ANIO'], dim_periodo, ['ANIO']),
    ('ING FK_Ubicacion (DEPT+PROV FILIAL)', ing_muestra.select([
        pl.col('DEPARTAMENTO_FILIAL').alias('DEPARTAMENTO'),
        pl.col('PROVINCIA_FILIAL').alias('PROVINCIA'),
    ]), ['DEPARTAMENTO', 'PROVINCIA'], dim_ubicacion, ['DEPARTAMENTO', 'PROVINCIA']),
]
for nombre, hecho, cols_fk, dim, cols_pk in checks:
    h = hecho.select(cols_fk).unique()
    d = dim.select(cols_pk).unique()
    huerfanos = h.join(d, on=cols_fk, how='anti').height
    print(f'  {nombre}: huérfanos = {huerfanos:,}')
print()

# B) Cardinalidad 1:N: cada clave de dimensión se repite en muchas filas de hecho
print('B) Cardinalidad 1:N (una dimensión → muchos hechos):')
for nombre, hecho, col in [
    ('DimUniversidad → FactMatriculados', mat_muestra, ['CODIGO_INEI']),
    ('DimPrograma → FactMatriculados', mat_muestra, ['CODIGO_SIU_PROGRAMA']),
    ('DimUbicacion → FactMatriculados', mat_ubic, ['DEPARTAMENTO', 'PROVINCIA']),
    ('DimLocal → FactMatriculados', mat_muestra, ['CODIGO_LOCAL']),
    ('DimPeriodo → FactMatriculados', mat_periodo, ['ANIO', 'SEMESTRE']),
    ('DimUniversidad → FactIngresantes', ing_muestra, ['CODIGO_INEI']),
    ('DimPeriodo → FactIngresantes', ing_muestra.select(pl.col('PROCESO_ESTANDARIZADO').cast(pl.Int64).alias('ANIO')), ['ANIO']),
]:
    agrup = hecho.group_by(col).len()
    max_por_clave = agrup.select(pl.col('len').max()).item()
    claves_con_muchos = agrup.filter(pl.col('len') > 1).height
    filas = agrup.select(pl.col('len').sum()).item()
    estado = '1:N OK' if claves_con_muchos > 0 else 'revisar'
    print(f'  {nombre}: {claves_con_muchos:,} claves con >1 hecho (máx {max_por_clave:,}) de {filas:,} filas → {estado}')
print()

# C) Grafo sin ciclos
print('C) Grafo de relaciones (sin ciclos):')
print('   · Todas las aristas son DIMENSIÓN → HECHO (1:N).')
print('   · No hay aristas entre dimensiones ni entre hechos.')
print('   · DimLocal solo se conecta a FactMatriculados.')
print('   → Modelo estrella puro, sin ciclos ni relaciones innecesarias.')
print('=' * 78)

VALIDACIÓN CONCEPTUAL DEL MODELO ESTRELLA
A) Integridad referencial de las FKs (sobre las muestras):
  FK_Universidad (CODIGO_INEI): huérfanos = 0
  FK_Programa (CODIGO_SIU_PROGRAMA): huérfanos = 0
  FK_Periodo (ANIO+SEMESTRE): huérfanos = 0
  FK_Ubicacion (DEPT+PROV): huérfanos = 0
  FK_Local (CODIGO_LOCAL): huérfanos = 0
  ING FK_Universidad (CODIGO_INEI): huérfanos = 0
  ING FK_Periodo (PROCESO_ESTANDARIZADO): huérfanos = 0
  ING FK_Ubicacion (DEPT+PROV FILIAL): huérfanos = 0

B) Cardinalidad 1:N (una dimensión → muchos hechos):
  DimUniversidad → FactMatriculados: 160 claves con >1 hecho (máx 11,555) de 99,999 filas → 1:N OK
  DimPrograma → FactMatriculados: 366 claves con >1 hecho (máx 5,843) de 99,999 filas → 1:N OK
  DimUbicacion → FactMatriculados: 88 claves con >1 hecho (máx 44,768) de 99,999 filas → 1:N OK
  DimLocal → FactMatriculados: 57 claves con >1 hecho (máx 49,456) de 99,999 filas → 1:N OK
  DimPeriodo → FactMatriculados: 12 claves con >1 hecho (máx 9,654) de 99,999 fi

---
## Conclusión y siguiente paso

La propuesta del **modelo estrella** para la capa Gold cubre las preguntas del caso (benchmark UCSP, top carreras en la región sur, análisis por universidad/año):

- **5 dimensiones** (Universidad, Programa, Periodo, Ubicación, Local) + **2 hechos** (FactIngresantes, FactMatriculados).
- Dimensiones **conformadas** (Universidad, Programa, Periodo, Ubicación) compartidas por ambos hechos; `DimLocal` exclusiva de Matriculados.
- Relaciones **1:N** verificadas y sin ciclos; claves naturales → FK directas.

**Siguiente paso (implementación Gold, fuera de este notebook):**
1. Generar dimensiones con **claves surrogate** (`SK_*`) y persistir como `data/Gold/dim_*.parquet`.
2. Generar hechos sustituyendo las claves naturales por FKs (`data/Gold/fact_*.parquet`).
3. Resolver los duplicados exactos de Matriculados (usar `matriculados_clean_v2.1.parquet` como fuente o `unique()` al construir el hecho).
4. Añadir estrategia SCD (Type 1/2) para atributos lentos (nombres de entidad, licenciamiento).
5. Documentar `ESTADO_LICENCIAMIENTO`: LICENCIADA → LICENCIADO; LEY DE CREACION se conserva; LICENCIA DENEGADA → NO LICENCIADO.